# ⚡ PHANTOM Cloud Hardware Testbed & Zero-Disk Benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FreakyAdy/phantom/blob/main/notebooks/phantom_cloud_tester.ipynb)

This notebook enables **100% free, zero-local-disk evaluation and reproduction** of the entire PHANTOM 14-model benchmark suite (spanning 135M to 72.7B parameters):
- **Free 15 GB Nvidia Cloud GPU** (T4 / L4)
- **100 GB Fast Ephemeral Cloud SSD** (0 bytes consumed on your laptop)
- **12.7 GB Host RAM**
- **1-Click Verification Across All 14 Evaluated Scale Models** (Qwen, DeepSeek, Llama, Mixtral, QwQ, Command-R, Yi)
- **Automated Publication-Ready Markdown & JSON Report Generation** (matching `docs/testing/TEMPLATE_TEST_REPORT.md`)


### Step 1: Environment Setup & Cloud Hardware Inspection
Clones the official PHANTOM repository, installs dependencies, and inspects live cloud GPU telemetry.

In [ ]:
# Clone repository if running inside Google Colab
import os, sys
if not os.path.exists('/content/phantom'):
    !git clone https://github.com/FreakyAdy/phantom.git /content/phantom
    %cd /content/phantom
else:
    %cd /content/phantom
    !git pull

# Install dependencies
!pip install -q -e python --no-deps
!pip install -q rich structlog huggingface_hub

# Install llama-cpp-python: prefer prebuilt CUDA 12.4 wheel (fast); fall back to plain wheel, then source build.
import subprocess, sys, os as _os
def _pip(arg, extra=None):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + (extra or []) + arg
    return subprocess.run(cmd, capture_output=True, text=True)

r = _pip(['llama-cpp-python'],
         ['--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu124', '--force-reinstall'])
print('prebuilt CUDA 12.4 wheel returncode:', r.returncode)
if r.returncode != 0:
    print('CUDA wheel unavailable; falling back to PyPI wheel (CPU):')
    r = _pip(['llama-cpp-python'])
    print('PyPI wheel returncode:', r.returncode)
if r.returncode != 0:
    print('Wheels failed; falling back to source build (CUDA, slow):')
    env = dict(_os.environ, CMAKE_ARGS='-DGGML_CUDA=on -DGGML_NATIVE=off', FORCE_CMAKE='1')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python'],
                       env=env, capture_output=True, text=True)
    print('source build returncode:', r.returncode)
if r.returncode != 0:
    print('---- pip stdout tail ----\n', r.stdout[-800:])
    print('---- pip stderr tail ----\n', r.stderr[-1500:])
else:
    import llama_cpp
    print('llama-cpp-python OK, version:', getattr(llama_cpp, '__version__', '?'))

# Hardware inspection


### Step 2: Interactive Model Selector & Hardware Profiler (1-Click Form)
Select any of the 14 verified models and target hardware configuration.
You can choose **Instant Virtual Profile** (simulates layer tiering and tok/s in milliseconds without downloading weights) or **Live Cloud Inference** (downloads model to ephemeral cloud scratch).

In [ ]:
#@title 🛠️ Select Model & Hardware Preset { run: "auto" }
MODEL_NAME = "qwen3-30b-a3b" #@param ["smollm-135m", "qwen2.5-coder-32b", "qwen3-30b-a3b", "llama-3-70b", "deepseek-r1-distill-qwen-32b", "deepseek-r1-distill-llama-70b", "mixtral-8x7b-instruct", "qwq-32b-preview", "qwen2.5-72b-instruct", "qwen2.5-32b-instruct", "deepseek-coder-33b-instruct", "codellama-70b-instruct", "command-r-35b", "yi-1.5-34b-chat"]
HARDWARE_PRESET = "colab-t4" #@param ["colab-t4", "rtx4050-laptop", "rtx4070-desktop", "rtx4090-desktop", "apple-m3-pro"]
MODE = "Virtual Simulation (Instant Zero-Disk Profile)" #@param ["Virtual Simulation (Instant Zero-Disk Profile)", "Live Cloud Inference (Download to Ephemeral Scratch)"]

print(f"Selected Model:    {MODEL_NAME}")
print(f"Hardware Preset:   {HARDWARE_PRESET}")
print(f"Execution Mode:    {MODE}")

# Run zero-disk profiler
!phantom profile {MODEL_NAME} --preset {HARDWARE_PRESET}


### Step 3: Run Automated Verification Battery & Generate Formal Report
Executes the automated verification battery:
1. **Algorithmic Dynamic Programming** (0/1 Knapsack, target 220)
2. **Mathematical Deduction** (Harmonic Mean Velocity, target 48 mph)
3. **Code Synthesis & Whitespace Normalization** (Word reversal)

Generates both a raw JSON telemetry record and a publication-ready Markdown report matching `docs/testing/TEMPLATE_TEST_REPORT.md`.

In [ ]:
import os
from pathlib import Path
from IPython.display import Markdown, display

is_dry = "Instant" in MODE
dry_flag = "--dry-run" if is_dry else ""

# Execute cloud runner harness
!python scripts/colab_runner.py --model {MODEL_NAME} --preset {HARDWARE_PRESET} {dry_flag} --output-dir /content/reports

# Display generated markdown report inline in notebook
reports_dir = Path("/content/reports")
md_reports = list(reports_dir.glob("*.md"))
if md_reports:
    latest_md = sorted(md_reports, key=lambda p: p.stat().st_mtime)[-1]
    print("=" * 68)
    print(f"  DISPLAYING GENERATED REPORT: {latest_md.name}")
    print("=" * 68)
    display(Markdown(latest_md.read_text(encoding="utf-8")))


### Step 4: Export Reports & Purge Cloud Scratch
Downloads the Markdown report and raw JSON results directly to your local computer,
then deletes any temporary model weights from Colab scratch drive (0 bytes remaining).

### Step 5: v2 Real Measurement — llama.cpp Native Draft Speculative Decode
Runs the authoritative real spec-decode benchmark on the selected model:
1. **Baseline** decode (no draft) and **native draft-model spec-decode** (default draft: `qwen2.5-0.5b`) via llama.cpp.
2. Configurable GPU layers (`--ngl`) and batch size (`--n-batch`) to sweep the CPU GEMM amortization.
3. Outputs real tok/s + speedup into `benchmarks/results/latest.json` and regenerates `RESULTS.md`.

Select **Live Cloud Inference** in Step 2 to download weights; **Virtual Simulation** runs the harness in dry-run (the `SKIPPED_LLAMA_CPP_NOT_INSTALLED` guard ensures no fabricated numbers).


In [ ]:
#@title ⚡ v2 Real Speculative-Decode Benchmark (llama.cpp native draft) { run: "auto" }
V2_MODEL = "qwen2.5-coder-32b"  #@param ["qwen2.5-coder-32b", "qwen3-30b-a3b"]
NGL = 14  #@param [0, 7, 14, 24, 999]
N_BATCH = 512  #@param [256, 512, 1024]

is_dry = "Instant" in MODE
iters = 2 if is_dry else 3
print(f"v2 spec-decode: Model={V2_MODEL} | ngl={NGL} | n_batch={N_BATCH} | iterations={iters} | Live={not is_dry}")

# Real llama.cpp native draft-model speculative decode (baseline vs draft) -> latest.json
!python benchmarks/run_real.py --focus spec-decode --model {V2_MODEL} --ngl {NGL} --n-batch {N_BATCH} --iterations {iters}

# Regenerate RESULTS.md from real measurements (one number, one source)
!python scripts/generate_results.py

print("v2 spec-decode results written to benchmarks/results/latest.json and RESULTS.md")


In [ ]:
import shutil
try:
    from google.colab import files
    is_colab = True
except ImportError:
    is_colab = False

reports_dir = Path("/content/reports")
if reports_dir.exists():
    for report_file in sorted(reports_dir.glob("*")):
        print(f"Ready for download: {report_file.name} ({report_file.stat().st_size} bytes)")
        if is_colab:
            files.download(str(report_file))

# Reclaim scratch disk
if os.path.exists("/content/scratch"):
    shutil.rmtree("/content/scratch")
    print("✓ Ephemeral cloud scratch drive cleaned. 0 bytes remaining on disk.")
else:
    print("✓ Clean environment: zero model weight files on disk.")
